## ⚡️ India Power Generation Region Prediction

Given *data about daily power generation in India*, let's try to predict what **region** a given report is from.

We will use eight different models to make our predictions.

Data source: https://www.kaggle.com/datasets/krishnadaskv/daily-power-generation-in-india-2013-2023

### Importing Libraries

In [63]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, GradientBoostingClassifier, RandomForestClassifier

In [18]:
data = pd.read_csv('archive/file_02.csv')
data

,index,Date,Region,Thermal Generation Actual (in MU),Thermal Generation Estimated (in MU),Nuclear Generation Actual (in MU),Nuclear Generation Estimated (in MU),Hydro Generation Actual (in MU),Hydro Generation Estimated (in MU)
0,0,2017-09-01,Northern,624.23,484.21,30.36,35.57,273.27,320.81
1,1,2017-09-01,Western,"1,106.89","1,024.33",25.17,3.81,72.00,21.53
2,2,2017-09-01,Southern,576.66,578.55,62.73,49.80,111.57,64.78
3,3,2017-09-01,Eastern,441.02,429.39,NaN,NaN,85.94,69.36
4,4,2017-09-01,NorthEastern,29.11,15.91,NaN,NaN,24.64,21.21
...,...,...,...,...,...,...,...,...,...
4940,305,2020-08-01,Northern,669.47,602.96,26.88,23.41,348.72,351.98
4941,306,2020-08-01,Western,"1,116.00","1,262.10",42.37,36.63,54.67,20.28
4942,307,2020-08-01,Southern,494.66,415.53,61.83,26.28,93.49,77.25
4943,308,2020-08-01,Eastern,482.86,547.03,NaN,NaN,87.22,93.78


In [19]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4945 entries, 0 to 4944
Data columns (total 9 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   index                                 4945 non-null   int64  
 1   Date                                  4945 non-null   object 
 2   Region                                4945 non-null   object 
 3   Thermal Generation Actual (in MU)     4945 non-null   object 
 4   Thermal Generation Estimated (in MU)  4945 non-null   object 
 5   Nuclear Generation Actual (in MU)     2967 non-null   float64
 6   Nuclear Generation Estimated (in MU)  2967 non-null   float64
 7   Hydro Generation Actual (in MU)       4945 non-null   float64
 8   Hydro Generation Estimated (in MU)    4945 non-null   float64
dtypes: float64(4), int64(1), object(4)
memory usage: 347.8+ KB


### Preprocessing

In [35]:
df = data.copy()

In [36]:
# Drop index column/checking missing values
df = df.drop('index', axis=1)

In [37]:
df

,Date,Region,Thermal Generation Actual (in MU),Thermal Generation Estimated (in MU),Nuclear Generation Actual (in MU),Nuclear Generation Estimated (in MU),Hydro Generation Actual (in MU),Hydro Generation Estimated (in MU)
0,2017-09-01,Northern,624.23,484.21,30.36,35.57,273.27,320.81
1,2017-09-01,Western,"1,106.89","1,024.33",25.17,3.81,72.00,21.53
2,2017-09-01,Southern,576.66,578.55,62.73,49.80,111.57,64.78
3,2017-09-01,Eastern,441.02,429.39,NaN,NaN,85.94,69.36
4,2017-09-01,NorthEastern,29.11,15.91,NaN,NaN,24.64,21.21
...,...,...,...,...,...,...,...,...
4940,2020-08-01,Northern,669.47,602.96,26.88,23.41,348.72,351.98
4941,2020-08-01,Western,"1,116.00","1,262.10",42.37,36.63,54.67,20.28
4942,2020-08-01,Southern,494.66,415.53,61.83,26.28,93.49,77.25
4943,2020-08-01,Eastern,482.86,547.03,NaN,NaN,87.22,93.78


In [38]:
df.isna().sum()

Date                                       0
Region                                     0
Thermal Generation Actual (in MU)          0
Thermal Generation Estimated (in MU)       0
Nuclear Generation Actual (in MU)       1978
Nuclear Generation Estimated (in MU)    1978
Hydro Generation Actual (in MU)            0
Hydro Generation Estimated (in MU)         0
dtype: int64

In [39]:
df.isna().mean() * 100

Date                                     0.0
Region                                   0.0
Thermal Generation Actual (in MU)        0.0
Thermal Generation Estimated (in MU)     0.0
Nuclear Generation Actual (in MU)       40.0
Nuclear Generation Estimated (in MU)    40.0
Hydro Generation Actual (in MU)          0.0
Hydro Generation Estimated (in MU)       0.0
dtype: float64

In [40]:
for column in ['Nuclear Generation Actual (in MU)', 'Nuclear Generation Estimated (in MU)']:
    df[column] = df[column].fillna(df[column].mean())

In [41]:
print("Total missing values:", df.isna().sum().sum())

Total missing values: 0


In [42]:
df

,Date,Region,Thermal Generation Actual (in MU),Thermal Generation Estimated (in MU),Nuclear Generation Actual (in MU),Nuclear Generation Estimated (in MU),Hydro Generation Actual (in MU),Hydro Generation Estimated (in MU)
0,2017-09-01,Northern,624.23,484.21,30.360000,35.570000,273.27,320.81
1,2017-09-01,Western,"1,106.89","1,024.33",25.170000,3.810000,72.00,21.53
2,2017-09-01,Southern,576.66,578.55,62.730000,49.800000,111.57,64.78
3,2017-09-01,Eastern,441.02,429.39,37.242208,36.987877,85.94,69.36
4,2017-09-01,NorthEastern,29.11,15.91,37.242208,36.987877,24.64,21.21
...,...,...,...,...,...,...,...,...
4940,2020-08-01,Northern,669.47,602.96,26.880000,23.410000,348.72,351.98
4941,2020-08-01,Western,"1,116.00","1,262.10",42.370000,36.630000,54.67,20.28
4942,2020-08-01,Southern,494.66,415.53,61.830000,26.280000,93.49,77.25
4943,2020-08-01,Eastern,482.86,547.03,37.242208,36.987877,87.22,93.78


In [43]:
# Creating year and month columns
df['Date'] = pd.to_datetime(df['Date'])

df['Year'] = df['Date'].apply(lambda x: x.year)
df['Month'] = df['Date'].apply(lambda x: x.month)

df = df.drop('Date', axis=1)

In [44]:
df

,Region,Thermal Generation Actual (in MU),Thermal Generation Estimated (in MU),Nuclear Generation Actual (in MU),Nuclear Generation Estimated (in MU),Hydro Generation Actual (in MU),Hydro Generation Estimated (in MU),Year,Month
0,Northern,624.23,484.21,30.360000,35.570000,273.27,320.81,2017,9
1,Western,"1,106.89","1,024.33",25.170000,3.810000,72.00,21.53,2017,9
2,Southern,576.66,578.55,62.730000,49.800000,111.57,64.78,2017,9
3,Eastern,441.02,429.39,37.242208,36.987877,85.94,69.36,2017,9
4,NorthEastern,29.11,15.91,37.242208,36.987877,24.64,21.21,2017,9
...,...,...,...,...,...,...,...,...,...
4940,Northern,669.47,602.96,26.880000,23.410000,348.72,351.98,2020,8
4941,Western,"1,116.00","1,262.10",42.370000,36.630000,54.67,20.28,2020,8
4942,Southern,494.66,415.53,61.830000,26.280000,93.49,77.25,2020,8
4943,Eastern,482.86,547.03,37.242208,36.987877,87.22,93.78,2020,8


In [45]:
# Removing commas from Thermal columns
for column in ['Thermal Generation Actual (in MU)', 'Thermal Generation Estimated (in MU)']:
    df[column] = df[column].apply(lambda x: float(x.replace(',', '')))

In [47]:
df.dtypes

Region                                   object
Thermal Generation Actual (in MU)       float64
Thermal Generation Estimated (in MU)    float64
Nuclear Generation Actual (in MU)       float64
Nuclear Generation Estimated (in MU)    float64
Hydro Generation Actual (in MU)         float64
Hydro Generation Estimated (in MU)      float64
Year                                      int64
Month                                     int64
dtype: object

#### Encoding Labels

In [50]:
label_encoder = LabelEncoder()

df['Region'] = label_encoder.fit_transform(df['Region'])

In [52]:
{index:label for index, label in enumerate(label_encoder.classes_)}

{0: 'Eastern', 1: 'NorthEastern', 2: 'Northern', 3: 'Southern', 4: 'Western'}

#### Splitting and scaling the data

In [53]:
y = df['Region'].copy()
X = df.drop('Region', axis=1).copy()

In [54]:
y

0       2
1       4
2       3
3       0
4       1
       ..
4940    2
4941    4
4942    3
4943    0
4944    1
Name: Region, Length: 4945, dtype: int64

In [55]:
X

,Thermal Generation Actual (in MU),Thermal Generation Estimated (in MU),Nuclear Generation Actual (in MU),Nuclear Generation Estimated (in MU),Hydro Generation Actual (in MU),Hydro Generation Estimated (in MU),Year,Month
0,624.23,484.21,30.360000,35.570000,273.27,320.81,2017,9
1,1106.89,1024.33,25.170000,3.810000,72.00,21.53,2017,9
2,576.66,578.55,62.730000,49.800000,111.57,64.78,2017,9
3,441.02,429.39,37.242208,36.987877,85.94,69.36,2017,9
4,29.11,15.91,37.242208,36.987877,24.64,21.21,2017,9
...,...,...,...,...,...,...,...,...
4940,669.47,602.96,26.880000,23.410000,348.72,351.98,2020,8
4941,1116.00,1262.10,42.370000,36.630000,54.67,20.28,2020,8
4942,494.66,415.53,61.830000,26.280000,93.49,77.25,2020,8
4943,482.86,547.03,37.242208,36.987877,87.22,93.78,2020,8


In [56]:
# Scale X
scaler = StandardScaler()

X = scaler.fit_transform(X)
X

array([[ 0.05280804, -0.23786479, -0.55945652, ...,  2.9739146 ,
        -1.73545914,  0.66194702],
       [ 1.31138889,  1.17108805, -0.98135303, ..., -0.67425516,
        -1.73545914,  0.66194702],
       [-0.07123516,  0.00822981,  2.07190954, ..., -0.14704538,
        -1.73545914,  0.66194702],
       ...,
       [-0.28505779, -0.4170229 ,  1.9987483 , ...,  0.00496169,
         1.67228888,  0.38073327],
       [-0.31582739, -0.07399302,  0.        , ...,  0.20645944,
         1.67228888,  0.38073327],
       [-1.4851764 , -1.41538332,  0.        , ..., -0.57149497,
         1.67228888,  0.38073327]], shape=(4945, 8))

In [57]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=1)

In [59]:
X_train.shape, X_test.shape

((3461, 8), (1484, 8))

### Modeling and Training

In [60]:
models = [
    LogisticRegression(),
    SVC(),
    MLPClassifier(),
    DecisionTreeClassifier(),
    AdaBoostClassifier(),
    BaggingClassifier(),
    GradientBoostingClassifier(),
    RandomForestClassifier()
]

model_names = [
    '         Logistic Regression',
    '      Support Vector Machine',
    '              Neural Network',
    '               Decision Tree',
    '        Ada Boost Classifier',
    '          Bagging Classifier',
    'Gradient Boosting Classifier',
    '    Random Forest Classifier'
]

In [62]:
results = []

for i in range(len(models)):
    models[i].fit(X_train, y_train)
    results.append(models[i].score(X_test, y_test))

### Results

In [64]:
for i in range(len(models)):
    print(model_names[i] + ": {:.4f}".format(results[i]))

         Logistic Regression: 0.9980
      Support Vector Machine: 0.9987
              Neural Network: 0.9987
               Decision Tree: 0.9993
        Ada Boost Classifier: 0.9993
          Bagging Classifier: 0.9993
Gradient Boosting Classifier: 0.9993
    Random Forest Classifier: 0.9987
